In [4]:
import os
print(os.path.expanduser('~/Desktop'))

C:\Users\13676/Desktop


In [6]:
import os
path = r'C:\Users\13676\Desktop\senna_cohort.csv'

print("文件是否存在:", os.path.exists(path))
print("文件大小 (字节):", os.path.getsize(path))

# 读取前 200 个字节看看原始内容
with open(path, 'rb') as f:
    raw = f.read(200)
    print("原始字节:", raw)

文件是否存在: True
文件大小 (字节): 10530068
原始字节: b'subject_id,hadm_id,gender,age,age_group,senna_exposed,index_time,alignment_type,senna_order_count,first_dose,first_dose_unit,baseline_k,baseline_k_charttime,hypokalemia_event,hypokalemia_time,hypokale'


In [9]:
import pandas as pd

path = r'C:\Users\13676\Desktop\senna_cohort.csv'

df = pd.read_csv(
    path,
    encoding='gb18030',        # 最全的中文编码，兼容 gbk
    encoding_errors='replace', # 万一还有坏字节，替换不崩
    low_memory=False
)

print(df.shape)
print(df.columns.tolist())
print(df.head(3))

(41529, 51)
['subject_id', 'hadm_id', 'gender', 'age', 'age_group', 'senna_exposed', 'index_time', 'alignment_type', 'senna_order_count', 'first_dose', 'first_dose_unit', 'baseline_k', 'baseline_k_charttime', 'hypokalemia_event', 'hypokalemia_time', 'hypokalemia_k_value', 'hypokalemia_48_72h', 'window_event_time', 'followup_hours', 'admittime', 'dischtime', 'deathtime', 'hospital_expire_flag', 'temporal_category', 'chf', 'ckd', 'dm', 'diuretic', 'acei_arb', 'digoxin', 'insulin', 'statin', 'anticoagulant', 'drug_count', 'polypharmacy', 'baseline_egfr', 'baseline_egfr_charttime', 'baseline_creatinine', 'lactulose', 'peg', 'bisacodyl', 'docusate', 'any_other_laxative', 'potassium_supplement', 'ppi', 'beta2_agonist', 'corticosteroid', 'amphotericin_b', 'diarrhea_vomiting', 'senna_total_dose', 'senna_days']
   subject_id   hadm_id gender  age age_group  senna_exposed           index_time alignment_type  senna_order_count first_dose first_dose_unit  baseline_k baseline_k_charttime  hypokalem

In [10]:
import pandas as pd
import numpy as np
from scipy import stats

df_main = df.dropna(subset=['senna_exposed', 'hypokalemia_event',
                             'baseline_k', 'age']).copy()

df_main['y'] = df_main['hypokalemia_event'].astype(int)
df_main['senna_exposed'] = df_main['senna_exposed'].astype(int)
df_main['gender_male'] = (df_main['gender'].str.upper().str[0] == 'M').astype(int)

print('主分析集行数:', len(df_main))
print('暴露组:', (df_main['senna_exposed'] == 1).sum(),
      f"({(df_main['senna_exposed']==1).mean()*100:.1f}%)")
print('对照组:', (df_main['senna_exposed'] == 0).sum(),
      f"({(df_main['senna_exposed']==0).mean()*100:.1f}%)")
print('低钾事件:', df_main['y'].sum(),
      f"({df_main['y'].mean()*100:.2f}%)")

主分析集行数: 41529
暴露组: 24226 (58.3%)
对照组: 17303 (41.7%)
低钾事件: 8752 (21.07%)


In [11]:
# 三分类 vs 结局
print('--- temporal_category × y ---')
print(pd.crosstab(df_main['temporal_category'], df_main['y'], dropna=False))

# 对照组低钾率
ctrl = df_main[df_main['senna_exposed'] == 0]
print(f"\n对照组低钾率: {ctrl['y'].mean()*100:.2f}% "
      f"({ctrl['y'].sum()}/{len(ctrl)})")

# 暴露组低钾率
exp = df_main[df_main['senna_exposed'] == 1]
print(f"暴露组低钾率: {exp['y'].mean()*100:.2f}% "
      f"({exp['y'].sum()}/{len(exp)})")

--- temporal_category × y ---
y                      0     1
temporal_category             
forward                0  5495
no_event           18731     0
unexposed          14046  3257

对照组低钾率: 18.82% (3257/17303)
暴露组低钾率: 22.68% (5495/24226)


In [13]:
# ---------- 第 1 步：先看类型 ----------
print('index_time dtype   :', df_main['index_time'].dtype)
print('admittime dtype    :', df_main['admittime'].dtype)
print('index_time 示例    :', df_main['index_time'].head(3).tolist())
print('admittime 示例     :', df_main['admittime'].head(3).tolist())

# ---------- 第 2 步：强制转 datetime（关键）----------
for c in ['index_time', 'admittime', 'dischtime',
          'baseline_k_charttime', 'hypokalemia_time',
          'window_event_time', 'deathtime',
          'baseline_egfr_charttime']:
    if c in df_main.columns:
        df_main[c] = pd.to_datetime(df_main[c], errors='coerce')

# 再确认一次
print('\n转换后:')
print('index_time dtype   :', df_main['index_time'].dtype)
print('admittime dtype    :', df_main['admittime'].dtype)

# 看有多少缺失（转换失败的）
print('\nNaT 数量:')
print('index_time NaT :', df_main['index_time'].isna().sum())
print('admittime NaT  :', df_main['admittime'].isna().sum())

index_time dtype   : object
admittime dtype    : object
index_time 示例    : ['23/10/2189 18:47:00', '22/3/2159 21:08:00', '3/4/2167 00:00:00']
admittime 示例     : ['21/10/2189 18:47:00', '20/3/2159 21:08:00', '1/4/2167 00:00:00']

转换后:
index_time dtype   : datetime64[ns]
admittime dtype    : datetime64[ns]

NaT 数量:
index_time NaT : 0
admittime NaT  : 0


In [3]:
import pandas as pd
import numpy as np
from scipy import stats

path = r'C:\Users\13676\Desktop\senna_cohort.csv'
df = pd.read_csv(path, encoding='gb18030',
                 encoding_errors='replace', low_memory=False)

time_cols = ['index_time', 'baseline_k_charttime', 'hypokalemia_time',
             'window_event_time', 'admittime', 'dischtime', 'deathtime',
             'baseline_egfr_charttime']
for c in time_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], dayfirst=True, errors='coerce')

# 主分析集
df_main = df.dropna(subset=['senna_exposed', 'hypokalemia_event',
                             'baseline_k', 'age']).copy()
df_main['y'] = df_main['hypokalemia_event'].astype(int)
df_main['senna_exposed'] = df_main['senna_exposed'].astype(int)
df_main['gender_male'] = (df_main['gender'].str.upper().str[0] == 'M').astype(int)

print('df_main 行数:', len(df_main))
print('暴露组:', (df_main['senna_exposed']==1).sum())
print('低钾事件:', df_main['y'].sum())

df_main 行数: 41529
暴露组: 24226
低钾事件: 8752


In [4]:
continuous_vars = [
    'age',
    'baseline_k',
    'baseline_creatinine',
    'baseline_egfr',
    'drug_count_fixed',
]

binary_vars = [
    'gender_male',
    'chf', 'ckd', 'dm',
    'diuretic', 'acei_arb', 'digoxin', 'insulin', 'statin', 'anticoagulant',
    'lactulose', 'peg', 'bisacodyl', 'docusate',
    'potassium_supplement', 'ppi', 'beta2_agonist',
    'corticosteroid', 'diarrhea_vomiting',
]

age_group_levels = ['65-74', '75-84', '85+']

label_map = {
    'age': 'Age, years',
    'baseline_k': 'Baseline serum potassium, mmol/L',
    'baseline_creatinine': 'Baseline creatinine, mg/dL',
    'baseline_egfr': 'Baseline eGFR, mL/min/1.73m²',
    'drug_count_fixed': 'Number of distinct concomitant drugs',
    'gender_male': 'Male sex, n (%)',
    'chf': 'Congestive heart failure, n (%)',
    'ckd': 'Chronic kidney disease, n (%)',
    'dm': 'Diabetes mellitus, n (%)',
    'diuretic': 'Diuretic use, n (%)',
    'acei_arb': 'ACEI/ARB use, n (%)',
    'digoxin': 'Digoxin use, n (%)',
    'insulin': 'Insulin use, n (%)',
    'statin': 'Statin use, n (%)',
    'anticoagulant': 'Anticoagulant use, n (%)',
    'lactulose': 'Lactulose use, n (%)',
    'peg': 'Polyethylene glycol use, n (%)',
    'bisacodyl': 'Bisacodyl use, n (%)',
    'docusate': 'Docusate use, n (%)',
    'potassium_supplement': 'Potassium supplement, n (%)',
    'ppi': 'Proton pump inhibitor, n (%)',
    'beta2_agonist': 'Beta-2 agonist, n (%)',
    'corticosteroid': 'Corticosteroid, n (%)',
    'diarrhea_vomiting': 'Diarrhea or vomiting, n (%)',
}

print('清单 OK')

清单 OK


In [5]:
exp = df_main[df_main['senna_exposed'] == 1].copy()
exp_vars = [v for v in ['senna_order_count', 'senna_total_dose', 'senna_days']
            if v in exp.columns]

for v in exp_vars:
    exp[v] = pd.to_numeric(exp[v], errors='coerce')

exp_desc = exp[exp_vars].describe(percentiles=[.25, .5, .75]).T
print('--- 暴露组番泻叶用药特征 (n = %d) ---' % len(exp))
print(exp_desc)
exp_desc.to_excel('TableS_exposure_characteristics.xlsx')
print('已保存 TableS_exposure_characteristics.xlsx')

--- 暴露组番泻叶用药特征 (n = 24226) ---
                     count       mean        std  min  25%  50%   75%  \
senna_order_count  24226.0   1.555395   0.973544  1.0  1.0  1.0   2.0   
senna_total_dose   24226.0  12.661038  14.207364  0.0  2.0  8.6  17.2   
senna_days         24226.0   1.252346   4.712456  0.0  0.0  0.0   0.0   

                          max  
senna_order_count   16.000000  
senna_total_dose   232.200000  
senna_days         154.541667  
已保存 TableS_exposure_characteristics.xlsx


In [8]:
# ========== 单元格 2：变量清单（最终版）==========
continuous_vars = [
    'age',
    'baseline_k',
    'baseline_creatinine',
    'baseline_egfr',
    'drug_count',              # ← 用修正后的 drug_count（均值≈13）
]

binary_vars = [
    'gender_male',
    'chf', 'ckd', 'dm',
    'diuretic', 'acei_arb', 'digoxin', 'insulin', 'statin', 'anticoagulant',
    # 4 个具体泻药（docusate 保留在表 1，主模型里剔除，见下）
    'lactulose', 'peg', 'bisacodyl', 'docusate',
    # 其余新增混杂
    'potassium_supplement', 'ppi', 'beta2_agonist',
    'corticosteroid', 'diarrhea_vomiting',
]

age_group_levels = ['65-74', '75-84', '85+']

label_map = {
    'age': 'Age, years',
    'baseline_k': 'Baseline serum potassium, mmol/L',
    'baseline_creatinine': 'Baseline creatinine, mg/dL',
    'baseline_egfr': 'Baseline eGFR, mL/min/1.73m²',
    'drug_count': 'Number of distinct concomitant drugs',
    'gender_male': 'Male sex, n (%)',
    'chf': 'Congestive heart failure, n (%)',
    'ckd': 'Chronic kidney disease, n (%)',
    'dm': 'Diabetes mellitus, n (%)',
    'diuretic': 'Diuretic use, n (%)',
    'acei_arb': 'ACEI/ARB use, n (%)',
    'digoxin': 'Digoxin use, n (%)',
    'insulin': 'Insulin use, n (%)',
    'statin': 'Statin use, n (%)',
    'anticoagulant': 'Anticoagulant use, n (%)',
    'lactulose': 'Lactulose use, n (%)',
    'peg': 'Polyethylene glycol use, n (%)',
    'bisacodyl': 'Bisacodyl use, n (%)',
    'docusate': 'Docusate use, n (%)',
    'potassium_supplement': 'Potassium supplement, n (%)',
    'ppi': 'Proton pump inhibitor, n (%)',
    'beta2_agonist': 'Beta-2 agonist, n (%)',
    'corticosteroid': 'Corticosteroid, n (%)',
    'diarrhea_vomiting': 'Diarrhea or vomiting, n (%)',
}

print('清单 OK')
print(f'  连续变量: {len(continuous_vars)}')
print(f'  二分类变量: {len(binary_vars)}')

清单 OK
  连续变量: 5
  二分类变量: 19


In [9]:
# ========== 单元格 3：暴露组用药特征 ==========
exp = df_main[df_main['senna_exposed'] == 1].copy()
exp_vars = [v for v in ['senna_order_count', 'senna_total_dose', 'senna_days']
            if v in exp.columns]

for v in exp_vars:
    exp[v] = pd.to_numeric(exp[v], errors='coerce')

exp_desc = exp[exp_vars].describe(percentiles=[.25, .5, .75]).T
print('--- 暴露组番泻叶用药特征 (n = %d) ---' % len(exp))
print(exp_desc)
exp_desc.to_excel('TableS_exposure_characteristics.xlsx')
print('已保存 TableS_exposure_characteristics.xlsx')

--- 暴露组番泻叶用药特征 (n = 24226) ---
                     count       mean        std  min  25%  50%   75%  \
senna_order_count  24226.0   1.555395   0.973544  1.0  1.0  1.0   2.0   
senna_total_dose   24226.0  12.661038  14.207364  0.0  2.0  8.6  17.2   
senna_days         24226.0   1.252346   4.712456  0.0  0.0  0.0   0.0   

                          max  
senna_order_count   16.000000  
senna_total_dose   232.200000  
senna_days         154.541667  
已保存 TableS_exposure_characteristics.xlsx


In [10]:
# ========== 单元格 4：基线表 Table 1 ==========
from scipy import stats

def is_normal(x, alpha=0.05):
    x = x.dropna()
    if len(x) > 5000:
        x = x.sample(5000, random_state=42)
    if len(x) < 3:
        return False
    return stats.shapiro(x).pvalue > alpha

def smd_continuous(x, g):
    x0, x1 = x[g == 0].dropna(), x[g == 1].dropna()
    if len(x0) < 2 or len(x1) < 2:
        return np.nan
    pooled = np.sqrt((x0.std()**2 + x1.std()**2) / 2)
    return np.abs(x1.mean() - x0.mean()) / pooled if pooled > 0 else np.nan

def smd_binary(x, g):
    p0, p1 = x[g == 0].mean(), x[g == 1].mean()
    pooled = np.sqrt((p0*(1-p0) + p1*(1-p1)) / 2)
    return np.abs(p1 - p0) / pooled if pooled > 0 else np.nan

def format_p(p):
    if pd.isna(p):
        return 'NA'
    if p < 1e-300:
        return '<1×10⁻³⁰⁰'
    if p >= 0.001:
        return f'{p:.3f}'
    e = int(np.floor(np.log10(p)))
    mant = p / (10 ** e)
    sup = str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    return f'{mant:.2f}×10{sup}'

def p_flag(p):
    return 'NA' if pd.isna(p) else ('Yes' if p < 0.05 else 'No')

g = df_main['senna_exposed']
rows = []

# 连续变量
for v in continuous_vars:
    if v not in df_main.columns:
        print(f'⚠️ 列不存在: {v}'); continue
    x = pd.to_numeric(df_main[v], errors='coerce')
    if is_normal(x):
        m0, s0 = x[g==0].mean(), x[g==0].std()
        m1, s1 = x[g==1].mean(), x[g==1].std()
        p = stats.ttest_ind(x[g==0].dropna(), x[g==1].dropna()).pvalue
        val0, val1 = f'{m0:.2f} ± {s0:.2f}', f'{m1:.2f} ± {s1:.2f}'
    else:
        q0 = x[g==0].quantile([.25, .5, .75]).values
        q1 = x[g==1].quantile([.25, .5, .75]).values
        p = stats.mannwhitneyu(x[g==0].dropna(), x[g==1].dropna()).pvalue
        val0 = f'{q0[1]:.2f} ({q0[0]:.2f}–{q0[2]:.2f})'
        val1 = f'{q1[1]:.2f} ({q1[0]:.2f}–{q1[2]:.2f})'
    rows.append({'Variable': label_map.get(v, v),
                 'Non-senna': val0, 'Senna': val1,
                 'SMD': round(smd_continuous(x, g), 3),
                 'P value': format_p(p), 'P<0.05': p_flag(p)})

# 二分类变量
for v in binary_vars:
    if v not in df_main.columns:
        print(f'⚠️ 列不存在: {v}'); continue
    x = pd.to_numeric(df_main[v], errors='coerce').fillna(0).astype(int)
    n0, n1 = (g==0).sum(), (g==1).sum()
    k0, k1 = int(x[g==0].sum()), int(x[g==1].sum())
    p = stats.chi2_contingency(pd.crosstab(x, g))[1]
    rows.append({'Variable': label_map.get(v, v),
                 'Non-senna': f'{k0} ({k0/n0*100:.1f})',
                 'Senna': f'{k1} ({k1/n1*100:.1f})',
                 'SMD': round(smd_binary(x, g), 3),
                 'P value': format_p(p), 'P<0.05': p_flag(p)})

# 年龄组
for cat in age_group_levels:
    x = (df_main['age_group'] == cat).astype(int)
    n0, n1 = (g==0).sum(), (g==1).sum()
    k0, k1 = int(x[g==0].sum()), int(x[g==1].sum())
    p = stats.chi2_contingency(pd.crosstab(x, g))[1]
    rows.append({'Variable': f'Age group: {cat}, n (%)',
                 'Non-senna': f'{k0} ({k0/n0*100:.1f})',
                 'Senna': f'{k1} ({k1/n1*100:.1f})',
                 'SMD': round(smd_binary(x, g), 3),
                 'P value': format_p(p), 'P<0.05': p_flag(p)})

# 结局行
n0, n1 = (g==0).sum(), (g==1).sum()
k0 = int(df_main.loc[g==0, 'y'].sum()); k1 = int(df_main.loc[g==1, 'y'].sum())
p = stats.chi2_contingency(pd.crosstab(df_main['y'], g))[1]
rows.append({'Variable': 'Incident hypokalemia, n (%)',
             'Non-senna': f'{k0} ({k0/n0*100:.1f})',
             'Senna': f'{k1} ({k1/n1*100:.1f})',
             'SMD': round(smd_binary(df_main['y'], g), 3),
             'P value': format_p(p), 'P<0.05': p_flag(p)})

table1 = pd.DataFrame(rows)
print(table1.to_string(index=False))
table1.to_excel('Table1_baseline.xlsx', index=False)
print(f'\n共 {len(table1)} 行，已保存 Table1_baseline.xlsx')

                            Variable           Non-senna               Senna   SMD    P value P<0.05
                          Age, years 75.00 (70.00–82.00) 77.00 (70.00–84.00) 0.112 5.23×10⁻²⁸    Yes
    Baseline serum potassium, mmol/L    4.10 (3.90–4.40)    4.20 (3.90–4.50) 0.174 3.44×10⁻⁵²    Yes
          Baseline creatinine, mg/dL    1.00 (0.80–1.30)    1.00 (0.80–1.40) 0.019      0.064     No
        Baseline eGFR, mL/min/1.73m² 69.63 (48.39–90.02) 68.35 (46.19–89.01) 0.048  1.01×10⁻⁶    Yes
Number of distinct concomitant drugs  11.00 (8.00–15.00) 14.00 (11.00–18.00) 0.578  <1×10⁻³⁰⁰    Yes
                     Male sex, n (%)         9157 (52.9)        12257 (50.6) 0.047  3.04×10⁻⁶    Yes
     Congestive heart failure, n (%)         4163 (24.1)         6704 (27.7) 0.083 1.61×10⁻¹⁶    Yes
       Chronic kidney disease, n (%)         4067 (23.5)         6104 (25.2) 0.039  8.14×10⁻⁵    Yes
            Diabetes mellitus, n (%)         5572 (32.2)         7610 (31.4) 0.017      0.0

In [11]:
# ========== 单元格 5：单变量 Logistic 回归 Table 2 ==========
import statsmodels.formula.api as smf

def format_p(p):
    if pd.isna(p): return 'NA'
    if p < 1e-300: return '<1×10⁻³⁰⁰'
    if p >= 0.001: return f'{p:.3f}'
    e = int(np.floor(np.log10(p)))
    mant = p / (10 ** e)
    sup = str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    return f'{mant:.2f}×10{sup}'

def p_flag(p):
    return 'NA' if pd.isna(p) else ('Yes' if p < 0.05 else 'No')

def univ_logit(df, y, x, x_display=None, categorical_levels=None):
    d = df[[y, x]].dropna().copy()
    d.columns = ['y', 'x']
    if categorical_levels is not None:
        d['x'] = pd.Categorical(d['x'], categories=categorical_levels, ordered=True)
        formula = 'y ~ C(x)'
    else:
        d['x'] = pd.to_numeric(d['x'], errors='coerce')
        d = d.dropna()
        formula = 'y ~ x'
    try:
        model = smf.logit(formula, data=d).fit(disp=0)
    except Exception as e:
        return [{'Variable': x_display or x, 'OR': np.nan,
                 'CI_low': np.nan, 'CI_high': np.nan,
                 'P': 'NA', 'P<0.05': 'NA'}]
    out = []
    for term in model.params.index:
        if term == 'Intercept':
            continue
        name = x_display or x
        if 'C(x)' in term:
            lv = term.split('T.')[1].rstrip(']')
            name = f'{x_display or x}: {lv} vs ref'
        p = model.pvalues[term]
        out.append({'Variable': name,
                    'OR': round(np.exp(model.params[term]), 3),
                    'CI_low': round(np.exp(model.conf_int().loc[term, 0]), 3),
                    'CI_high': round(np.exp(model.conf_int().loc[term, 1]), 3),
                    'P': format_p(p),
                    'P<0.05': p_flag(p)})
    return out

uni_results = []
uni_results.extend(univ_logit(df_main, 'y', 'senna_exposed',
                               x_display='Senna exposure (1 vs 0)'))

for v in continuous_vars:
    if v in df_main.columns:
        uni_results.extend(univ_logit(df_main, 'y', v,
                                       x_display=label_map.get(v, v)))

for v in binary_vars:
    if v in df_main.columns:
        uni_results.extend(univ_logit(df_main, 'y', v,
                                       x_display=label_map.get(v, v)))

if 'age_group' in df_main.columns:
    uni_results.extend(univ_logit(df_main, 'y', 'age_group',
                                   x_display='Age group',
                                   categorical_levels=age_group_levels))

table2 = pd.DataFrame(uni_results)
print(table2.to_string(index=False))
table2.to_excel('Table2_univariate_logistic.xlsx', index=False)
print(f'\n共 {len(table2)} 行，已保存 Table2_univariate_logistic.xlsx')

                            Variable     OR  CI_low  CI_high           P P<0.05
             Senna exposure (1 vs 0)  1.265   1.205    1.328  2.21×10⁻²¹    Yes
                          Age, years  1.002   0.999    1.005       0.186     No
    Baseline serum potassium, mmol/L  0.377   0.355    0.400 1.56×10⁻²²⁶    Yes
          Baseline creatinine, mg/dL  0.990   0.969    1.012       0.383     No
        Baseline eGFR, mL/min/1.73m²  1.001   1.000    1.002       0.064     No
Number of distinct concomitant drugs  1.103   1.099    1.108   <1×10⁻³⁰⁰    Yes
                     Male sex, n (%)  0.779   0.743    0.817  4.83×10⁻²⁵    Yes
     Congestive heart failure, n (%)  1.364   1.295    1.437  5.83×10⁻³²    Yes
       Chronic kidney disease, n (%)  0.856   0.809    0.905   5.39×10⁻⁸    Yes
            Diabetes mellitus, n (%)  0.891   0.846    0.938   9.89×10⁻⁶    Yes
                 Diuretic use, n (%)  1.896   1.807    1.989 1.29×10⁻¹⁴⁹    Yes
                 ACEI/ARB use, n (%)  0.